# Notebook 4 — Grafos, PageRank y Algoritmo Pregel

**Proyecto:** Sistema Web de Gestion de Contratacion Docente — EMI Cochabamba

## Que hace este notebook

1. Construccion del grafo bipartito docente↔asignatura con pesos reales (monto)
2. PageRank sobre el grafo para identificar docentes mas influyentes
3. Algoritmo Pregel implementado desde cero con multiprocessing real
4. Analisis de centralidad del grafo
5. Visualizacion del grafo con NetworkX

**Pregel:** modelo de computacion de grafos iterativo por supersteps. Cada nodo envia mensajes a sus vecinos, recibe mensajes, actualiza su estado. Se repite hasta convergencia. PageRank es una aplicacion directa de Pregel.

In [ ]:
!pip install pandas networkx matplotlib seaborn unidecode -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import networkx as nx
import warnings
import multiprocessing as mp
import time
from collections import defaultdict
from unidecode import unidecode

warnings.filterwarnings('ignore')

COLORES = {
    'docente':     '#1a4fa0',
    'asignatura':  '#16a34a',
    'primario':    '#1a4fa0',
    'secundario':  '#2563eb',
    'acento':      '#16a34a',
    'peligro':     '#c0392b',
    'advertencia': '#d97706',
    'morado':      '#7c3aed',
}
PALETA = list(COLORES.values())

plt.rcParams['figure.dpi']        = 120
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False

print(f'CPUs disponibles: {mp.cpu_count()}')

## 1. Carga del dataset

In [ ]:
from google.colab import files
print('Sube emi_contratos_limpio.csv')
files.upload()

In [ ]:
df = pd.read_csv('emi_contratos_limpio.csv')

def inferir_area(asignatura):
    a = unidecode(str(asignatura).lower())
    if any(k in a for k in ['red', 'administracion de red']): return 'REDES'
    if any(k in a for k in ['base de dato']): return 'BASES_DE_DATOS'
    if any(k in a for k in ['program', 'python', 'java', 'software', 'web']): return 'PROGRAMACION'
    if any(k in a for k in ['inteligencia']): return 'INTELIGENCIA_ARTIFICIAL'
    if any(k in a for k in ['seguridad', 'forense']): return 'SEGURIDAD'
    if any(k in a for k in ['sistema operativo', 'arquitectura', 'digital']): return 'SISTEMAS'
    if any(k in a for k in ['calculo', 'algebra', 'estadistica', 'ecuacion', 'fisica', 'probabilidad', 'estocastico']): return 'MATEMATICAS'
    if any(k in a for k in ['estructura de dato', 'algoritmo']): return 'ALGORITMOS'
    return 'GESTION'

if 'AREA' not in df.columns:
    df['AREA'] = df['ASIGNATURA'].apply(inferir_area)

print(f'Dataset: {len(df)} contratos, {df["CEDULA"].nunique()} docentes, {df["ASIGNATURA"].nunique()} asignaturas')

## 2. Construccion del grafo bipartito

Nodos tipo D (docentes) y tipo A (asignaturas). Aristas ponderadas por monto del contrato.

In [ ]:
G = nx.Graph()

docentes_info = df.groupby('CEDULA').agg(
    nombre=('NOMBRE', 'first'),
    grado=('GRADO', 'first'),
    n_contratos=('NRO', 'count'),
    monto_total=('MONTO', 'sum'),
).reset_index()

for _, row in docentes_info.iterrows():
    G.add_node(f"D_{row['CEDULA']}", tipo='docente',
               nombre=row['nombre'], grado=row['grado'],
               n_contratos=int(row['n_contratos']), monto_total=float(row['monto_total']))

asig_info = df.groupby('ASIGNATURA').agg(
    area=('AREA', 'first'), n_contratos=('NRO', 'count'), monto_total=('MONTO', 'sum')
).reset_index()

for _, row in asig_info.iterrows():
    G.add_node(f"A_{row['ASIGNATURA']}", tipo='asignatura',
               nombre=row['ASIGNATURA'], area=row['area'],
               n_contratos=int(row['n_contratos']), monto_total=float(row['monto_total']))

for _, row in df.iterrows():
    nd = f"D_{row['CEDULA']}"
    na = f"A_{row['ASIGNATURA']}"
    if G.has_edge(nd, na):
        G[nd][na]['weight']   += float(row['MONTO'])
        G[nd][na]['contratos'] += 1
    else:
        G.add_edge(nd, na, weight=float(row['MONTO']), contratos=1, modalidad=row['MODALIDAD'])

nodos_doc = [n for n, d in G.nodes(data=True) if d['tipo'] == 'docente']
nodos_asi = [n for n, d in G.nodes(data=True) if d['tipo'] == 'asignatura']

print(f'Grafo bipartito:')
print(f'  Nodos docentes    : {len(nodos_doc)}')
print(f'  Nodos asignaturas : {len(nodos_asi)}')
print(f'  Aristas           : {G.number_of_edges()}')
print(f'  Es bipartito      : {nx.is_bipartite(G)}')

## 3. PageRank con NetworkX (referencia)

In [ ]:
pagerank_nx = nx.pagerank(G, alpha=0.85, weight='weight', max_iter=100, tol=1e-6)

pr_docentes    = {n: v for n, v in pagerank_nx.items() if n.startswith('D_')}
pr_asignaturas = {n: v for n, v in pagerank_nx.items() if n.startswith('A_')}

top_pr = sorted(pr_docentes.items(), key=lambda x: x[1], reverse=True)[:15]
df_pr = pd.DataFrame(top_pr, columns=['nodo', 'pagerank'])
df_pr['cedula'] = df_pr['nodo'].str.replace('D_', '')
df_pr = df_pr.merge(docentes_info[['CEDULA','nombre','grado','n_contratos','monto_total']],
                    left_on='cedula', right_on='CEDULA', how='left')
df_pr['pagerank_pct'] = (df_pr['pagerank'] * 100).round(4)

print('Top 15 Docentes por PageRank (ponderado por monto):')
print(df_pr[['nombre','grado','n_contratos','monto_total','pagerank_pct']].to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ax = axes[0]
top15 = df_pr.head(15)
nombre_corto = top15['nombre'].apply(lambda x: x.split(',')[0].strip() if ',' in str(x) else str(x)[:20])
colors = ['#1a4fa0' if i < 3 else '#2563eb' if i < 7 else '#93c5fd' for i in range(len(top15))]
bars = ax.barh(nombre_corto[::-1], top15['pagerank_pct'][::-1], color=colors[::-1], alpha=0.9)
for bar, val in zip(bars, top15['pagerank_pct'][::-1]):
    ax.text(val + 0.001, bar.get_y() + bar.get_height()/2, f'{val:.4f}%', va='center', fontsize=8)
ax.set_title('Top 15 Docentes por PageRank', fontweight='bold')
ax.set_xlabel('PageRank Score (%)')

ax = axes[1]
top_asig_pr = sorted(pr_asignaturas.items(), key=lambda x: x[1], reverse=True)[:12]
nombres_asig = [n.replace('A_','') for n, _ in top_asig_pr]
scores_asig  = [v*100 for _, v in top_asig_pr]
bars = ax.barh(nombres_asig[::-1], scores_asig[::-1], color=COLORES['acento'], alpha=0.85)
for bar, val in zip(bars, scores_asig[::-1]):
    ax.text(val + 0.001, bar.get_y() + bar.get_height()/2, f'{val:.4f}%', va='center', fontsize=8)
ax.set_title('Top 12 Asignaturas por PageRank', fontweight='bold')
ax.set_xlabel('PageRank Score (%)')
ax.tick_params(axis='y', labelsize=8)

plt.tight_layout()
plt.savefig('pagerank_ranking.png', bbox_inches='tight', dpi=150)
plt.show()

## 4. Algoritmo Pregel — PageRank iterativo con multiprocessing

Pregel procesa el grafo en supersteps iterativos:
- Cada nodo activo envia mensajes a sus vecinos
- Los nodos reciben mensajes y actualizan su PageRank
- Se repite hasta convergencia (diferencia < epsilon)
- Cada superstep se ejecuta en paralelo con multiprocessing

In [ ]:
def pregel_superstep(args):
    """
    Un superstep de Pregel para un nodo.
    PR(v) = (1-d)/N + d * sum(PR(u) * peso(u,v) / grado_total(u))
    """
    nodo, vecinos_con_pesos, pageranks_actuales, d, N = args
    suma = 0.0
    for vecino, peso_arista, grado_vecino in vecinos_con_pesos:
        if vecino in pageranks_actuales and grado_vecino > 0:
            suma += pageranks_actuales[vecino] * (peso_arista / grado_vecino)
    nuevo_pr = (1 - d) / N + d * suma
    return (nodo, nuevo_pr)


def pagerank_pregel(grafo, d=0.85, max_iter=50, epsilon=1e-6, n_workers=None):
    """
    PageRank con modelo Pregel y multiprocessing real.
    Retorna (pageranks, historial_convergencia, n_iteraciones)
    """
    if n_workers is None:
        n_workers = mp.cpu_count()

    nodos = list(grafo.nodes())
    N     = len(nodos)
    pageranks = {n: 1.0 / N for n in nodos}

    # Precomputar estructura del grafo
    estructura = {}
    for nodo in nodos:
        vecinos     = list(grafo.neighbors(nodo))
        grado_total = sum(grafo[nodo][v].get('weight', 1) for v in vecinos)
        estructura[nodo] = [(v, grafo[nodo][v].get('weight', 1), grado_total) for v in vecinos]

    historial = []
    t_inicio  = time.perf_counter()

    for superstep in range(max_iter):
        t_step = time.perf_counter()

        args_workers = [(nodo, estructura[nodo], dict(pageranks), d, N) for nodo in nodos]

        with mp.Pool(processes=n_workers) as pool:
            resultados = pool.map(pregel_superstep, args_workers)

        nuevos_pageranks = dict(resultados)
        diferencia = sum(abs(nuevos_pageranks[n] - pageranks[n]) for n in nodos)

        t_dur = (time.perf_counter() - t_step) * 1000
        historial.append({'superstep': superstep+1, 'diferencia': diferencia, 'tiempo_ms': t_dur})

        pageranks = nuevos_pageranks

        if superstep % 5 == 0 or diferencia < epsilon:
            print(f'  Superstep {superstep+1:>3}: delta={diferencia:.8f}  |  {t_dur:.1f}ms')

        if diferencia < epsilon:
            print(f'\n  Convergencia en superstep {superstep+1}')
            break

    t_total = (time.perf_counter() - t_inicio) * 1000
    print(f'\n  Tiempo total Pregel: {t_total:.1f}ms | {n_workers} workers')
    return pageranks, pd.DataFrame(historial), superstep + 1


print('Algoritmo Pregel implementado')

In [ ]:
print('='*60)
print('EJECUTANDO PAGERANK CON PREGEL')
print(f'Grafo: {G.number_of_nodes()} nodos, {G.number_of_edges()} aristas')
print('='*60)

pr_pregel, df_convergencia, n_iter = pagerank_pregel(G, d=0.85, max_iter=50, epsilon=1e-6)

# Validar contra NetworkX
mae = np.mean([abs(pr_pregel[n] - pagerank_nx[n]) for n in G.nodes()])
print(f'\nValidacion vs NetworkX PageRank:')
print(f'  Error Medio Absoluto (MAE): {mae:.2e}')
print(f'  Equivalentes: {"SI" if mae < 1e-4 else "Revisar"}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.semilogy(df_convergencia['superstep'], df_convergencia['diferencia'],
            'o-', color=COLORES['primario'], linewidth=2.5, markersize=5)
ax.axhline(1e-6, color=COLORES['peligro'], linestyle='--', linewidth=1.5, label='Umbral convergencia (1e-6)')
ax.fill_between(df_convergencia['superstep'], df_convergencia['diferencia'], 1e-6, alpha=0.1, color=COLORES['primario'])
ax.set_title('Convergencia del Algoritmo Pregel\n(norma L1 por superstep)', fontweight='bold')
ax.set_xlabel('Superstep')
ax.set_ylabel('Delta PageRank (log)')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.bar(df_convergencia['superstep'], df_convergencia['tiempo_ms'],
       color=COLORES['secundario'], alpha=0.8, width=0.7)
ax.axhline(df_convergencia['tiempo_ms'].mean(), color=COLORES['peligro'], linestyle='--', linewidth=1.5,
           label=f"Media: {df_convergencia['tiempo_ms'].mean():.1f}ms")
ax.set_title('Tiempo por Superstep (Pregel paralelo)', fontweight='bold')
ax.set_xlabel('Superstep')
ax.set_ylabel('Tiempo (ms)')
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('pregel_convergencia.png', bbox_inches='tight', dpi=150)
plt.show()

## 5. Visualizacion del grafo

In [ ]:
# Top 20 docentes por PageRank para visualizacion
top_doc_pr  = sorted(pr_pregel.items(), key=lambda x: x[1], reverse=True)
top_doc_ids = [n for n, _ in top_doc_pr if n.startswith('D_')][:20]
asig_ids    = [n for n in G.nodes() if n.startswith('A_')]
G_sub       = G.subgraph(set(top_doc_ids + asig_ids)).copy()

pos = nx.spring_layout(G_sub, weight='weight', k=2.0, seed=42, iterations=100)

fig, ax = plt.subplots(figsize=(18, 12))
ax.set_facecolor('#f8f9fb')
fig.patch.set_facecolor('#f8f9fb')

nodos_tipo_doc  = [n for n in G_sub.nodes() if G_sub.nodes[n]['tipo'] == 'docente']
nodos_tipo_asig = [n for n in G_sub.nodes() if G_sub.nodes[n]['tipo'] == 'asignatura']

sizes_doc  = [pr_pregel.get(n, 0) * 80000 + 300 for n in nodos_tipo_doc]
sizes_asig = [pr_pregel.get(n, 0) * 80000 + 500 for n in nodos_tipo_asig]

max_peso    = max((G_sub[u][v]['weight'] for u, v in G_sub.edges()), default=1)
edge_widths = [G_sub[u][v]['weight'] / max_peso * 3 + 0.5 for u, v in G_sub.edges()]

nx.draw_networkx_edges(G_sub, pos, ax=ax, width=edge_widths, alpha=0.4, edge_color='#94a3b8')
nx.draw_networkx_nodes(G_sub, pos, nodelist=nodos_tipo_doc, ax=ax,
                       node_size=sizes_doc, node_color=COLORES['docente'], alpha=0.85,
                       linewidths=1.5, edgecolors='white')
nx.draw_networkx_nodes(G_sub, pos, nodelist=nodos_tipo_asig, ax=ax,
                       node_size=sizes_asig, node_color=COLORES['asignatura'], alpha=0.85,
                       node_shape='s', linewidths=1.5, edgecolors='white')

labels_asig = {n: G_sub.nodes[n]['nombre'].replace(' ', '\n') for n in nodos_tipo_asig}
nx.draw_networkx_labels(G_sub, pos, labels=labels_asig, ax=ax, font_size=6, font_color='white', font_weight='bold')

leyenda = [
    mpatches.Patch(color=COLORES['docente'],    label=f'Docentes (top 20 PageRank, n={len(nodos_tipo_doc)})'),
    mpatches.Patch(color=COLORES['asignatura'], label=f'Asignaturas (n={len(nodos_tipo_asig)})'),
]
ax.legend(handles=leyenda, loc='upper left', fontsize=10, framealpha=0.9)
ax.set_title('Grafo Bipartito Docente - Asignatura — EMI Cochabamba 2026\nTamano proporcional a PageRank (Pregel) | Grosor de arista proporcional al monto',
             fontsize=13, fontweight='bold', pad=15)
ax.axis('off')

plt.tight_layout()
plt.savefig('grafo_docentes.png', bbox_inches='tight', dpi=150)
plt.show()
print('Grafo guardado')

## 6. Benchmark Pregel secuencial vs paralelo

In [ ]:
def pagerank_secuencial(grafo, d=0.85, max_iter=50, epsilon=1e-6):
    nodos = list(grafo.nodes())
    N = len(nodos)
    pr = {n: 1.0/N for n in nodos}
    for _ in range(max_iter):
        nuevos = {}
        for nodo in nodos:
            vecinos     = list(grafo.neighbors(nodo))
            grado_total = sum(grafo[nodo][v].get('weight',1) for v in vecinos)
            suma = sum(pr[v] * grafo[nodo][v].get('weight',1) / max(grado_total,1) for v in vecinos)
            nuevos[nodo] = (1 - d)/N + d * suma
        diff = sum(abs(nuevos[n] - pr[n]) for n in nodos)
        pr   = nuevos
        if diff < epsilon:
            break
    return pr


n_rep = 3
t0 = time.perf_counter()
for _ in range(n_rep):
    pagerank_secuencial(G)
t_seq = (time.perf_counter() - t0) / n_rep * 1000

t0 = time.perf_counter()
for _ in range(n_rep):
    pagerank_pregel(G, max_iter=n_iter)
t_par = (time.perf_counter() - t0) / n_rep * 1000

speedup = t_seq / t_par if t_par > 0 else 1
print('='*50)
print(f'  Tiempo secuencial : {t_seq:.1f} ms')
print(f'  Tiempo Pregel     : {t_par:.1f} ms')
print(f'  Speedup           : {speedup:.2f}x')
print(f'  Workers usados    : {mp.cpu_count()}')
print(f'  Iteraciones Pregel: {n_iter}')
print('='*50)

## 7. Exportar resultados

In [ ]:
df_pr_export = pd.DataFrame([
    {'cedula': n.replace('D_',''), 'pagerank': v, 'pagerank_pct': v*100}
    for n, v in pr_pregel.items() if n.startswith('D_')
]).sort_values('pagerank', ascending=False)

df_pr_export = df_pr_export.merge(
    docentes_info[['CEDULA','nombre','grado','n_contratos','monto_total']],
    left_on='cedula', right_on='CEDULA', how='left'
)
df_pr_export.to_csv('pagerank_docentes.csv', index=False, encoding='utf-8-sig')
df_convergencia.to_csv('pregel_convergencia.csv', index=False, encoding='utf-8-sig')

from google.colab import files
for archivo in ['pagerank_docentes.csv', 'pregel_convergencia.csv',
                'grafo_docentes.png', 'pregel_convergencia.png', 'pagerank_ranking.png']:
    files.download(archivo)

print('Resultados exportados')
print()
print('Top 10 docentes por PageRank (Pregel):')
print(df_pr_export[['nombre','grado','n_contratos','monto_total','pagerank_pct']].head(10).round(4).to_string(index=False))